# 14f — Pull PTS (Political Terror Scale)

**Source:** Gibney, Cornett, Wood, Haschke, Arnon, Pisanò, Souza & Barnes —
*Political Terror Scale*, 1976–2023  
**Provider:** politicalterrorscale.org  
**Coverage:** ~180 countries, 1976–present, annual  
**Access:** Direct Excel download from politicalterrorscale.org (no registration)

## What this notebook does

1. Downloads the PTS Excel file from politicalterrorscale.org
2. Selects and renames the three coding-source columns plus derived variables
3. Writes a country-year parquet to ADLS

## Why PTS is needed

The Political Terror Scale codes **government repression** on an ordinal 1–5 scale,
based on three independent sources:

- **PTS-A** — Amnesty International annual report
- **PTS-S** — US State Department Country Reports on Human Rights
- **PTS-H** — Human Rights Watch World Report

PTS appears as a predictor in the Harff (2003) genocide model, the Goldstone (2010)
political instability model, and several mass-killing risk assessments. The synthesis
document identifies `pts_max` (maximum across sources) as a key predictor missing
from current acquisition.

High PTS scores (4–5) indicate systematic terror against civilians; transitions
from 3→4 or 4→5 are used as a leading indicator for mass killing onset.

## PTS Scale

| Score | Meaning |
|---|---|
| 1 | Countries under a secure rule of law; political murders extremely rare |
| 2 | Limited amounts of imprisonment for nonviolent activity; torture occurs |
| 3 | Extensive political imprisonment; executions, political murders common |
| 4 | Civil and political rights violations extended to large numbers of the population |
| 5 | Terror has expanded to the whole population; leaders place no limits on the means |

## Variables produced

| Variable | Description | Use |
|---|---|---|
| `pts_a` | PTS score from Amnesty International | Predictor |
| `pts_s` | PTS score from US State Department | Predictor |
| `pts_h` | PTS score from Human Rights Watch | Predictor |
| `pts_mean` | Mean of available source scores | Predictor |
| `pts_max` | Maximum across available sources | **Primary predictor** (Harff 2003) |
| `pts_high` | 1 if pts_max ≥ 4 (systematic terror) | **Dependent variable** |
| `pts_escalation` | 1 if pts_max increased ≥1 point vs. prior year | **Dependent variable** |

## Output path on ADLS

```
raw/pts/{RUN_DATE}/pts_annual.parquet
```

Add `"pts": "raw/pts"` to `RAW_PREFIXES` in notebook 14.

In [ ]:
%%capture
%pip install requests pandas openpyxl pyarrow azure-identity adlfs --quiet

In [ ]:
import io
import os
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import requests
import adlfs
from azure.identity import DefaultAzureCredential

warnings.filterwarnings('ignore')
RUN_DATE = datetime.utcnow().strftime('%Y%m%d')

In [ ]:
ADLS_ACCOUNT_NAME = os.environ['ADLS_ACCOUNT_NAME']
ADLS_CONTAINER    = os.getenv('ADLS_CONTAINER', 'data')
CROSSWALK_PATH    = '../data/country_crosswalk.csv'

# PTS publishes an annual Excel file. URL pattern as of 2024/2025:
# https://www.politicalterrorscale.org/Data/Docs/PTS-2024.xlsx
# The filename increments with each release year. Try recent years in order.
PTS_URLS = [
    'https://www.politicalterrorscale.org/Data/Docs/PTS-2024.xlsx',
    'https://www.politicalterrorscale.org/Data/Docs/PTS-2023.xlsx',
    'https://www.politicalterrorscale.org/Data/Docs/PTS-2022.xlsx',
]

# PTS scores ≥ this threshold are coded as systematic high terror
PTS_HIGH_THRESHOLD = 4

In [ ]:
credential = DefaultAzureCredential()
storage_options = {'account_name': ADLS_ACCOUNT_NAME, 'credential': credential}

def adls_path(subpath):
    return f'abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/{subpath}'

def write_parquet(df, subpath):
    df.to_parquet(adls_path(subpath), storage_options=storage_options,
                  index=False, engine='pyarrow')
    print(f'  Written {len(df):,} rows → {subpath}')

In [ ]:
def _download_pts(url, timeout=60):
    print(f'Downloading PTS from {url} ...')
    resp = requests.get(url, timeout=timeout,
                        headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    xl = pd.ExcelFile(io.BytesIO(resp.content))
    print(f'  Sheets: {xl.sheet_names}')
    # PTS workbooks typically have a single data sheet
    data_sheet = next(
        (s for s in xl.sheet_names
         if any(k in s.lower() for k in ['pts', 'data', 'country'])),
        xl.sheet_names[0],
    )
    print(f'  Reading sheet: {data_sheet}')
    return pd.read_excel(xl, sheet_name=data_sheet)


df_raw = None
for url in PTS_URLS:
    try:
        df_raw = _download_pts(url)
        print(f'  Raw shape: {df_raw.shape}')
        print(f'  Columns  : {df_raw.columns.tolist()[:20]}')
        break
    except Exception as exc:
        print(f'  Failed: {exc}')

if df_raw is None:
    raise RuntimeError(
        'Could not download PTS data. '
        'Download manually from https://www.politicalterrorscale.org/Data/ '
        'and load with: df_raw = pd.read_excel("PTS-2024.xlsx")'
    )

In [ ]:
# Normalise column names — PTS column naming is consistent across releases
df = df_raw.copy()
df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]
print('Columns:', df.columns.tolist())

# PTS releases a long-format panel: Country, Year, PTS_A, PTS_S, PTS_H, COuntryCode, ...
# Map to standard names
rename_map = {}
for src, tgt in [
    ('country',          'country_name'),
    ('countryname',      'country_name'),
    ('country_name',     'country_name'),
    ('year',             'year'),
    ('pts_a',            'pts_a'),
    ('amnesty',          'pts_a'),
    ('pts-a',            'pts_a'),
    ('pts_s',            'pts_s'),
    ('state',            'pts_s'),
    ('state_department', 'pts_s'),
    ('pts-s',            'pts_s'),
    ('pts_h',            'pts_h'),
    ('hrw',              'pts_h'),
    ('human_rights_watch', 'pts_h'),
    ('pts-h',            'pts_h'),
    # ISO codes — PTS includes both ISO2 and ISO3 in some releases
    ('iso3',             'iso3'),
    ('iso3c',            'iso3'),
    ('ccode',            'ccode'),
    ('cowcode',          'ccode'),
    ('countrycode',      'iso3'),
    ('country_code',     'iso3'),
]:
    if src in df.columns and src != tgt:
        rename_map[src] = tgt

df = df.rename(columns=rename_map)
print('After rename:', df.columns.tolist()[:15])

In [ ]:
# If ISO3 is not directly available, map via crosswalk using country name
from pathlib import Path

cw_path = Path(CROSSWALK_PATH)
if not cw_path.exists():
    cw_path = Path('data/country_crosswalk.csv')

df_cw = pd.read_csv(cw_path, dtype=str)
name_to_iso3 = dict(zip(df_cw['country_name_canonical'].str.lower(), df_cw['iso3']))

# PTS-specific name overrides
pts_name_overrides = {
    'united states':              'USA',
    'united kingdom':             'GBR',
    'russia':                     'RUS',
    'iran':                       'IRN',
    'south korea':                'KOR',
    'north korea':                'PRK',
    'syria':                      'SYR',
    'vietnam':                    'VNM',
    'laos':                       'LAO',
    'democratic republic of congo':  'COD',
    'dr congo':                   'COD',
    'republic of congo':          'COG',
    'congo (brazzaville)':        'COG',
    'congo (kinshasa)':           'COD',
    "ivory coast":                'CIV',
    "cote d'ivoire":              'CIV',
    'bolivia':                    'BOL',
    'venezuela':                  'VEN',
    'tanzania':                   'TZA',
    'moldova':                    'MDA',
    'timor-leste':                'TLS',
    'east timor':                 'TLS',
    'eswatini':                   'SWZ',
    'swaziland':                  'SWZ',
    'north macedonia':            'MKD',
    'cape verde':                 'CPV',
    'czechia':                    'CZE',
    'czech republic':             'CZE',
}

def _resolve_iso3(row):
    # If iso3 already present and looks valid, use it
    if 'iso3' in row.index and pd.notna(row.get('iso3')) and len(str(row['iso3'])) == 3:
        return str(row['iso3']).upper()
    # Fallback: name lookup
    if 'country_name' in row.index and pd.notna(row.get('country_name')):
        n = str(row['country_name']).strip().lower()
        return pts_name_overrides.get(n) or name_to_iso3.get(n)
    return None

df['iso3'] = df.apply(_resolve_iso3, axis=1)

unmatched = df[df['iso3'].isna()]['country_name'].unique() if 'country_name' in df.columns else []
print(f'Unmatched countries ({len(unmatched)}): {sorted(str(u) for u in unmatched)[:15]}')

In [ ]:
# Clean and select columns
df = df.dropna(subset=['iso3', 'year'])
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df = df.dropna(subset=['year'])
df['year'] = df['year'].astype(int)

for col in ['pts_a', 'pts_s', 'pts_h']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Compute cross-source summary columns
score_cols = [c for c in ['pts_a', 'pts_s', 'pts_h'] if c in df.columns]
df['pts_mean'] = df[score_cols].mean(axis=1, skipna=True)
df['pts_max']  = df[score_cols].max(axis=1, skipna=True)

print(f'Rows: {len(df):,}')
print(f'Years: {df["year"].min()}–{df["year"].max()}')
print(f'Countries: {df["iso3"].nunique()}')
if 'pts_max' in df.columns:
    print('pts_max distribution:\n', df['pts_max'].value_counts().sort_index().to_string())

In [ ]:
# Derive dependent-variable flags
df = df.sort_values(['iso3', 'year']).reset_index(drop=True)

# High terror: pts_max ≥ 4 in this year
df['pts_high'] = (df['pts_max'] >= PTS_HIGH_THRESHOLD).astype('Int64')

# Escalation: pts_max increased ≥1 point vs. prior year (within country)
df['pts_max_prev'] = df.groupby('iso3')['pts_max'].shift(1)
df['pts_escalation'] = (
    (df['pts_max'] - df['pts_max_prev'] >= 1).astype('Int64')
)
df = df.drop(columns=['pts_max_prev'])

final_cols = [c for c in
    ['iso3', 'year', 'pts_a', 'pts_s', 'pts_h', 'pts_mean', 'pts_max',
     'pts_high', 'pts_escalation']
    if c in df.columns]
df = df[final_cols]

print(f'Final panel: {len(df):,} country-years, {df["iso3"].nunique()} countries')
if 'pts_high' in df.columns:
    print(f'pts_high base rate     : {df["pts_high"].mean():.2%}')
if 'pts_escalation' in df.columns:
    print(f'pts_escalation base rate: {df["pts_escalation"].mean():.2%}')
df.head()

In [ ]:
write_parquet(df, f'raw/pts/{RUN_DATE}/pts_annual.parquet')

print()
print('=' * 55)
print('Notebook 14f complete')
print('=' * 55)
print(f'  Rows      : {len(df):,}')
print(f'  Countries : {df["iso3"].nunique()}')
print(f'  Years     : {df["year"].min()}–{df["year"].max()}')
print()
print('Dependent variables written:')
print('  pts_high        — 1 if pts_max ≥ 4 (systematic state terror)')
print('  pts_escalation  — 1 if pts_max increased ≥1 point vs. prior year')
print()
print('Predictors: pts_a, pts_s, pts_h, pts_mean, pts_max')
print()
print('Next: add  "pts": "raw/pts"  to RAW_PREFIXES in notebook 14.')